#### Hypothesentest: Anteilswerte (Binomialtest)

In diesem Test prüfen wir, ob eine behauptete Erfolgswahrscheinlichkeit (oder Defektquote) der Realität standhält.

##### 1. Das Szenario
Ein Hersteller behauptet, dass höchstens **10 %** seiner Produkte defekt sind. Wir haben den Verdacht, dass die Quote höher ist ("Untertreibung").

**Die Daten:**
* Behauptete Quote ($p$): $0,10$
* Stichprobenumfang ($n$): $20$ Stück
* Beobachtete Defekte ($k$): $5$ Stück

##### 2. Hypothesen aufstellen
Wir formulieren die Hypothesen immer so, dass die "Gegenbehauptung" in die $H_1$ kommt:

* **$H_0$ (Nullhypothese):** $p \leq 0,10$
    *(Alles okay, der Hersteller sagt die Wahrheit.)*
* **$H_1$ (Alternativhypothese):** $p > 0,10$
    *(Die Ausschussquote ist in Wahrheit höher – der Hersteller lügt.)*

**Signifikanzniveau ($\alpha$):** $0,05$ ($5\ \%$).
Das ist unser Risiko: Die Wahrscheinlichkeit, den Hersteller fälschlicherweise als Lügner zu bezeichnen, obwohl er die Wahrheit sagt (Fehler 1. Art).

##### 3. Testlogik & p-Wert
Wir berechnen den **p-Wert**. Das ist die Wahrscheinlichkeit, bei einer wahren Quote von $10\ \%$ zufällig $5$ oder mehr Defekte in $20$ Stücken zu finden.

$$P(X \geq 5) = P(X=5) + P(X=6) + \dots + P(X=20)$$
Oder einfacher über die Gegenwahrscheinlichkeit:
$$P(X \geq 5) = 1 - P(X \leq 4)$$


In [ ]:
from scipy.stats import binomtest
result = binomtest(k=5, n=20, p=0.1, alternative='greater')
print(f"Der p-Wert ist: {result.pvalue:.4f}")

alpha = 0.05
if result.pvalue < alpha:
    print("Ergebnis signifikant: H0 ablehnen! Der Hersteller flunkert vermutlich.")
else:
    print("Nicht signifikant: H0 beibehalten. Könnte noch Zufall sein.")

#### 🔋 Beispiel: Der Akku-Check (Gauß-Test / Z-Test)

In diesem Test prüfen wir, ob ein Premium-Hersteller bei der Kapazitätsangabe seiner Akkus schummelt. Da wir die Standardabweichung der Grundgesamtheit ($\sigma$) kennen, nutzen wir den **Z-Test**.

#### 1. Die Ausgangslage
* **Behaupteter Mittelwert ($\mu_0$):** $3000 \text{ mAh}$
* **Bekannte Standardabweichung ($\sigma$):** $50 \text{ mAh}$
* **Stichprobenumfang ($n$):** $25$ Akkus
* **Gemessener Durchschnitt ($\bar{x}$):** $2980 \text{ mAh}$

#### 2. Der Test-Aufbau
Wir führen einen **einseitigen Test** durch, da uns nur interessiert, ob die Kapazität *zu niedrig* ist.

* **$H_0$ (Nullhypothese):** $\mu \geq 3000$ (Versprechen gehalten)
* **$H_1$ (Alternativhypothese):** $\mu < 3000$ (Mogelpackung)
* **Signifikanzniveau ($\alpha$):** $0,05$

#### 3. Berechnung der Testgröße (Z-Wert)
Der Z-Wert gibt an, wie viele Standardfehler unser Ergebnis vom behaupteten Mittelwert entfernt ist:

$$Z = \frac{\bar{x} - \mu_0}{\sigma / \sqrt{n}} = \frac{2980 - 3000}{50 / \sqrt{25}} = \frac{-20}{10} = -2$$

Unser Ergebnis liegt also genau **2 Standardabweichungen** unter dem versprochenen Wert.

#### 4. Entscheidung
Bei einem einseitigen Test mit $\alpha = 0,05$ liegt der kritische Wert bei **$-1,645$**.

* **Unser Z-Wert:** $-2$
* **Kritischer Bereich:** Alles links von $-1,645$.

Da $-2 < -1,645$ liegt unser Ergebnis im Ablehnungsbereich.


#### 💡 Fazit
Die Wahrscheinlichkeit, einen Durchschnitt von $2980 \text{ mAh}$ oder weniger rein durch Zufall zu erhalten (p-Wert), liegt bei ca. **$2,28 \%$**. Da dies unter unserem Limit von $5 \%$ liegt, lehnen wir $H_0$ ab.
**Ergebnis:** Wir werfen dem Hersteller offiziell Etikettenschwindel vor!

In [ ]:
import numpy as np
from scipy.stats import norm

# 1. Parameter definieren
mu_0 = 3000
sigma = 50
n = 25
x_bar = 2980
alpha = 0.05

# 2. Teststatistik berechnen (Z-Wert)
standard_fehler = sigma / np.sqrt(n)
z_score = (x_bar - mu_0) / standard_fehler

# 3. Kritischen Wert und p-Wert bestimmen
# norm.ppf(alpha) gibt den Z-Wert an, unter dem alpha Prozent der Fläche liegen
kritischer_wert = norm.ppf(alpha)
p_wert = norm.cdf(z_score)

# 4. Ausgabe der Ergebnisse
print(f"--- Gauß-Test Ergebnis ---")
print(f"Z-Score:          {z_score:.2f}")
print(f"Kritischer Wert:  {kritischer_wert:.3f}")
print(f"p-Wert:           {p_wert:.4f} (entspricht {p_wert*100:.2f}%)")
print(f"--------------------------")

if z_score <= kritischer_wert:
    print("Entscheidung: H0 ablehnen! (Mogelpackung)")
else:
    print("Entscheidung: H0 beibehalten. (Im Rahmen der Toleranz)")


In [ ]:
# Scipy hat kein Gauß-test!!!
# Aber man kann es selber schnell schreiben:

def gauss_test(x_bar, mu_0, sigma, n, alternative='smaller'):
    z = (x_bar - mu_0) / (sigma / np.sqrt(n))
    if alternative == 'smaller':
        p = norm.cdf(z)
    elif alternative == 'greater':
        p = norm.sf(z)
    else: # zweiseitig
        p = 2 * norm.cdf(-abs(z))

    return z, p

_, pvalue = gauss_test(x_bar, mu_0, sigma, n, alternative='smaller')

alpha = 0.05
if pvalue < alpha:
    print("Ergebnis signifikant: H0 ablehnen! Der Hersteller flunkert vermutlich.")
else:
    print("Nicht signifikant: H0 beibehalten. Könnte noch Zufall sein.")

#### 📉 Beispiel: Ein-Stichproben-$t$-Test (Akkulaufzeit)

In diesem Test prüfen wir eine Mittelwerts-Behauptung, wenn wir die wahre Standardabweichung der Welt ($\sigma$) **nicht** kennen und sie aus unserer kleinen Stichprobe schätzen müssen.

#### 1. Die Situation
Ein Hersteller behauptet: „Unsere Akkus halten im Schnitt **500 Stunden**.“ Du kaufst 10 Akkus ($n=10$) und misst folgende Werte:
`480, 490, 505, 485, 470, 495, 480, 510, 475, 480`

#### 2. Hypothesen & Niveau
* **$H_0$ (Nullhypothese):** $\mu \geq 500$ (Versprechen wird eingehalten)
* **$H_1$ (Alternativhypothese):** $\mu < 500$ (Mogelpackung!)
* **Signifikanzniveau ($\alpha$):** $0,05$ (einseitiger Test nach unten)

#### 3. Daten berechnen
* **Mittelwert ($\bar{x}$):** $487,0$ h
* **Soll-Wert ($\mu_0$):** $500$ h
* **Stichprobenstandardabweichung ($s$):**
  $$s = \sqrt{\frac{\sum (x_i - \bar{x})^2}{n-1}} \approx 12,95$$

#### 4. Den t-Wert berechnen
Der t-Wert zeigt uns, wie viele geschätzte Standardfehler wir vom Soll-Wert entfernt sind:
$$t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}} = \frac{487 - 500}{\frac{12,95}{\sqrt{10}}} \approx -3,17$$

#### 5. Entscheidung
* **Freiheitsgrade ($df$):** $10 - 1 = 9$
* **Kritischer Wert:** Bei $df=9$ und $\alpha=0,05$ liegt die Grenze bei **$-1,833$**.
* **Vergleich:** Unser Wert ($-3,17$) ist viel extremer als der kritische Wert.

**Ergebnis:** $H_0$ wird abgelehnt. Die Wahrscheinlichkeit, so ein Ergebnis bei echten 500h zu erhalten, ist winzig ($p \approx 0,0057$). Der Hersteller hat geflunkert.

In [ ]:
import numpy as np
from scipy.stats import ttest_1samp

# 1. Daten eingeben
akkus = [480, 490, 505, 485, 470, 495, 480, 510, 475, 480]
pop_mean = 500 # Der abehauptete Wert

# 2. t-Test durchführen
# alternative='less', weil wir testen, ob die Akkus WENIGER halten (H1: mu < 500)
t_stat, p_val = ttest_1samp(akkus, popmean=pop_mean, alternative='less')

# 3. Manuelle Berechnung zur Kontrolle
n = len(akkus)
x_bar = np.mean(akkus)
s = np.std(akkus, ddof=1) # Wichtig: ddof=1 für Stichprobe
t_manuell = (x_bar - pop_mean) / (s / np.sqrt(n))

# 4. Ausgabe
print(f"--- Ein-Stichproben t-Test ---")
print(f"Mittelwert (x_bar):   {x_bar:.1f} h")
print(f"Std.-Abw. (s):        {s:.2f} h")
print(f"t-Statistik:          {t_stat:.3f}")
print(f"t-Wert manuell:       {t_manuell:.3f}")
print(f"p-Wert:               {p_val:.4f}")
print(f"------------------------------")

if p_val < 0.05:
    print("Entscheidung: H0 ablehnen! (Signifikant kürzer)")
else:
    print("Entscheidung: H0 beibehalten.")

#### ⚖️ Beispiel: Der zweiseitige t-Test (Maschinen-Vergleich)

Beim zweiseitigen Test interessiert uns nicht, ob ein Wert nur *kleiner* oder nur *größer* ist, sondern ob er sich **überhaupt signifikant** vom Sollwert unterscheidet.

##### 1. Das Szenario
Du hast eine neue Abfüllmaschine. Du willst wissen: Befüllt sie die Kartons **anders** als die alte (Sollwert: 500g)?
* Du weißt nicht, ob sie zu viel oder zu wenig einfüllt.
* Jede starke Abweichung (in beide Richtungen) ist für dich ein Signal.

**Die Daten ($n=5$):** `[495, 502, 498, 501, 499]`

##### 2. Hypothesen
* **$H_0$ (Nullhypothese):** $\mu = 500$ (Die Maschine füllt exakt 500g ab).
* **$H_1$ (Alternativhypothese):** $\mu \neq 500$ (Es gibt einen Unterschied – egal in welche Richtung).

##### 3. Logik: Die geteilte Gefahr ($\alpha/2$)
Wir wählen ein Signifikanzniveau von $\alpha = 0,05$ (5 %). Da wir zweiseitig testen, verteilen wir dieses Risiko auf beide Enden der Glockenkurve:
* **2,5 %** Risiko für "viel zu leicht".
* **2,5 %** Risiko für "viel zu schwer".

Dadurch rücken die kritischen Werte (die "Hürden") weiter nach außen. Es ist also **schwerer**, $H_0$ abzulehnen, als bei einem einseitigen Test.

##### 4. Entscheidung
Wir berechnen den $t$-Wert wie gewohnt. In der Tabelle schauen wir jedoch unter $\alpha/2 = 0,025$ nach. Nur wenn unser $t$-Wert extrem hoch (positiv) oder extrem niedrig (negativ) ist, lehnen wir die Nullhypothese ab.

In [ ]:
from scipy.stats import ttest_1samp
import numpy as np

# 1. Daten und Sollwert
daten = [495, 502, 498, 501, 499]
sollwert = 500

# 2. Ein-Stichproben-t-Test (zweiseitig)
# alternative='two-sided' ist default, wir schreiben es zur Deutlichkeit hin
t_stat, p_val = ttest_1samp(daten, popmean=sollwert, alternative='two-sided')

# 3. Mittelwert der Stichprobe berechnen
mittelwert = np.mean(daten)

print(f"Ergebnis des zweiseitigen t-Tests:")
print(f"----------------------------------")
print(f"Stichproben-Mittelwert: {mittelwert:.2f} g")
print(f"t-Statistik:            {t_stat:.4f}")
print(f"p-Wert:                 {p_val:.4f}")

# 4. Interpretation
alpha = 0.05
if p_val < alpha:
    print(f"\nFazit: p < {alpha} -> H0 ablehnen!")
    print("Die Maschine befüllt die Kartons signifikant anders.")
else:
    print(f"\nFazit: p > {alpha} -> H0 beibehalten.")
    print("Der Unterschied zum Sollwert ist rein zufällig.")

#### ⚖️ Beispiel: Der Wilcoxon-Vorzeichen-Rangtest

Der Wilcoxon-Test ist die „goldene Mitte“: Er ist robust gegen Ausreißer (wie der Vorzeichentest), nutzt aber zusätzlich die Information, wie weit die Werte vom Soll-Wert entfernt sind (ähnlich wie der t-Test).

##### 1. Die Situation
Wir testen $n=6$ Akkus. Der versprochene Median liegt bei **500 h**.
Unsere Messwerte: `497, 503, 505, 485, 502, 515`

##### 2. Die „Gewichtsklasse“ berechnen
Zuerst schauen wir uns die Differenzen zum Soll-Wert an und vergeben Ränge für die **absoluten** Abstände:

| Wert | Differenz ($x - 500$) | Absoluter Abstand | Rang |
| :--- | :--- | :--- | :--- |
| 502 | $+2$ | $2$ | **1** |
| 497 | $-3$ | $3$ | **2,5** (geteilter Rang) |
| 503 | $+3$ | $3$ | **2,5** (geteilter Rang) |
| 505 | $+5$ | $5$ | **4** |
| 485 | $-15$ | $15$ | **5,5** (geteilter Rang) |
| 515 | $+15$ | $15$ | **5,5** (geteilter Rang) |

##### 3. Summen der Ränge
Jetzt trennen wir die positiven von den negativen Rängen:

* **$W^+$ (Summe der positiven Ränge):** $1 + 2,5 + 4 + 5,5 = 13$
* **$W^-$ (Summe der negativen Ränge):** $2,5 + 5,5 = 8$

Die Teststatistik $W$ ist der kleinere der beiden Werte: **$W = 8$**.

##### 4. Das Urteil
Bei $n=6$ und $\alpha = 0,05$ (zweiseitig) müsste $W \leq 0$ sein, um signifikant zu sein.
Da $8 > 0$, können wir die Nullhypothese **nicht ablehnen**.


In [ ]:
import numpy as np
from scipy.stats import wilcoxon

# 1. Daten und Soll-Wert
akkus = np.array([497, 503, 505, 485, 502, 515])
soll_wert = 500

# 2. Differenzen berechnen
differenzen = akkus - soll_wert

# 3. Wilcoxon-Test durchführen
# Wir testen zweiseitig (default), ob der Median ungleich 500 ist.
# mode='exact' wird bei kleinen Stichproben (n < 25) empfohlen.
res = wilcoxon(differenzen, mode='exact')

# 4. Ergebnisse ausgeben
print(f"--- Wilcoxon-Vorzeichen-Rangtest ---")
print(f"Teststatistik (W): {res.statistic}")
print(f"p-Wert:            {res.pvalue:.4f}")
print(f"------------------------------------")

if res.pvalue < 0.05:
    print("Entscheidung: H0 ablehnen! Der Median ist signifikant anders.")
else:
    print("Entscheidung: H0 beibehalten. Der Median liegt im Bereich von 500h.")

#### 🎲 Beispiel: Ist der Würfel gezinkt? (Chi-Quadrat-Test)

Wir testen, ob ein Würfel fair ist oder ob das Casino nachgeholfen hat. Bei einem fairen Würfel sollte jede Zahl mit der gleichen Wahrscheinlichkeit ($1/6$) erscheinen.

##### 1. Hypothesen & Versuchsaufbau
* **$H_0$:** Der Würfel ist fair ($p_1 = p_2 = \dots = p_6 = 1/6$).
* **$H_1$:** Der Würfel ist gezinkt (mindestens eine Wahrscheinlichkeit weicht ab).
* **Stichprobe ($n$):** 60 Würfe.
* **Signifikanzniveau ($\alpha$):** 0,05.

##### 2. Beobachtete vs. Erwartete Werte
Bei 60 Würfen erwarten wir für jede Zahl genau $60 / 6 = 10$ Treffer.

| Würfelzahl ($i$) | Beobachtet ($O_i$) | Erwartet ($E_i$) | Abweichung $\frac{(O_i - E_i)^2}{E_i}$ |
| :--- | :--- | :--- | :--- |
| 1 | 5 | 10 | $(5-10)^2 / 10 = 2,5$ |
| 2 | 8 | 10 | $(8-10)^2 / 10 = 0,4$ |
| 3 | 9 | 10 | $(9-10)^2 / 10 = 0,1$ |
| 4 | 8 | 10 | $(8-10)^2 / 10 = 0,4$ |
| 5 | 10 | 10 | $(10-10)^2 / 10 = 0,0$ |
| 6 | 20 | 10 | $(20-10)^2 / 10 = 10,0$ |
| **Summe** | **60** | **60** | **$\chi^2_{stat} = 13,4$** |

##### 3. Die Entscheidung
* **Freiheitsgrade ($df$):** Kategorien $- 1 = 6 - 1 = 5$.
* **Kritischer Wert:** In der $\chi^2$-Tabelle finden wir für $df=5$ und $\alpha=0,05$ den Wert **11,07**.

**Vergleich:**
Unser berechneter Wert ($13,4$) ist größer als der kritische Wert ($11,07$).

**Fazit:** Die Nullhypothese wird abgelehnt. Die Abweichungen (besonders die häufige "6") sind zu stark, um noch als reiner Zufall durchzugehen. Wir haben statistisch signifikante Evidenz, dass der Würfel gezinkt ist.

In [ ]:
import numpy as np
from scipy.stats import chisquare, chi2

# 1. Beobachtete Werte (unsere 60 Würfe)
beobachtet = np.array([5, 8, 9, 8, 10, 20])

# 2. Erwartete Werte (bei fairem Würfel und n=60)
# Wenn wir f_exp nicht angeben, nimmt SciPy automatisch eine Gleichverteilung an.
erwartet = np.array([10, 10, 10, 10, 10, 10])

# 3. Chi-Quadrat-Test durchführen
chi2_stat, p_val = chisquare(f_obs=beobachtet, f_exp=erwartet)

# 4. Kritischen Wert berechnen (für die manuelle Prüfung)
alpha = 0.05
df = len(beobachtet) - 1
kritischer_wert = chi2.ppf(1 - alpha, df)

# 5. Ausgabe
print(f"--- Chi-Quadrat-Anpassungstest ---")
print(f"Chi2-Statistik:      {chi2_stat:.2f}")
print(f"Kritischer Wert (5%): {kritischer_wert:.2f}")
print(f"p-Wert:              {p_val:.4f}")
print(f"----------------------------------")

if chi2_stat > kritischer_wert:
    print("Entscheidung: H0 ablehnen! Der Würfel ist gezinkt.")
else:
    print("Entscheidung: H0 beibehalten. Der Würfel erscheint fair.")

#### 🪙 Beispiel: Der Münzwurf-Marathon (Binomial-Anpassungstest)

Wir wollen prüfen, ob ein Experiment mit 100 Teilnehmern tatsächlich einer fairen Binomialverteilung folgt. Jeder Teilnehmer wirft eine Münze **3-mal** ($n=3$). Wir zählen, wie oft "Kopf" erscheint.

#### 1. Die Ausgangslage
* **Anzahl Versuche pro Person ($n$):** 3
* **Anzahl Teilnehmer ($N$):** 100
* **Behauptete Erfolgswahrscheinlichkeit ($p$):** 0,5 (faire Münze)

**Hypothesen:**
* **$H_0$:** Die Daten sind binomialverteilt mit $p=0,5$.
* **$H_1$:** Die Daten folgen dieser Verteilung nicht.

#### 2. Erwartete Werte berechnen
Die Wahrscheinlichkeiten für $0, 1, 2, 3$ Köpfe bei $n=3$ sind:
* $P(X=0) = 0,125$
* $P(X=1) = 0,375$
* $P(X=2) = 0,375$
* $P(X=3) = 0,125$

Multipliziert mit 100 Teilnehmern ergeben sich die erwarteten Häufigkeiten ($E_i$).

| Köpfe ($k$) | Beobachtet ($O_k$) | Erwartet ($E_k$) |
| :--- | :--- | :--- |
| 0 | 15 | 12,5 |
| 1 | 35 | 37,5 |
| 2 | 40 | 37,5 |
| 3 | 10 | 12,5 |

In [ ]:
import numpy as np
from scipy.stats import binom, chisquare

# 1. Beobachtete Daten: 100 Personen warfen 3x die Münze
# Index 0: 0x Kopf, Index 1: 1x Kopf, etc.
beobachtet = np.array([15, 35, 40, 10])
n_versuche = 3
N_teilnehmer = 100
p_hypothese = 0.5

# 2. Erwartete Häufigkeiten berechnen (Binomial-Logik)
k_werte = np.arange(0, n_versuche + 1)
# Wahrscheinlichkeiten: [0.125, 0.375, 0.375, 0.125]
probs = binom.pmf(k_werte, n_versuche, p_hypothese)
erwartet = probs * N_teilnehmer

# 3. Chi-Quadrat-Test
# ddof=0, da wir p=0.5 fest vorgegeben haben (nicht aus Daten geschätzt)
chi2_stat, p_val = chisquare(f_obs=beobachtet, f_exp=erwartet, ddof=0)

# 4. Ergebnisse
print(f"--- Chi-Quadrat-Test auf Binomialverteilung ---")
print(f"Beobachtet: {beobachtet}")
print(f"Erwartet:   {erwartet}")
print(f"Chi2-Stat:  {chi2_stat:.3f}")
print(f"p-Wert:     {p_val:.4f}")
print(f"-----------------------------------------------")

if p_val > 0.05:
    print("Fazit: H0 bleibt bestehen. Die Münzwürfe wirken bibnomialverteilt.")
else:
    print("Fazit: H0 abgelehnt. Die Verteilung weicht signifikant ab.")

Wichtiger Hinweis zu ddof:
* ddof=0: Du nimmst diesen Wert, wenn du testest: "Folgen die Daten genau $p=0,5$?". Das ist ein Test auf Fairness.
* ddof=1: Du nimmst diesen Wert, wenn du sagst: "Ich weiß nicht, wie groß $p$ ist. Ich schätze $p$ erst aus meinen 100 Beobachtungen und will nur wissen, ob es überhaupt irgendeine Binomialverteilung ist." Durch das Schätzen von $p$ verlierst du einen Freiheitsgrad.

#### 📈 Der Q-Q Plot: Der visuelle Verteilungs-Check

Ein Q-Q Plot vergleicht die Quantile deiner Daten mit den Quantilen einer theoretischen Verteilung (z. B. der Normalverteilung).

### Wie man ihn liest:
* **Gerade Linie:** Deine Daten folgen der Verteilung perfekt.
* **Abweichungen an den Enden:** Deine Daten haben "fette Enden" (Ausreißer) oder sind schief.
* **S-Kurve:** Die Wölbung (Kurtosis) deiner Daten passt nicht zur gewählten Verteilung.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# 1. Beispiel-Daten (100 Werte, normalverteilt)
np.random.seed(42)
daten = np.random.normal(loc=100, scale=15, size=100)

# 2. Q-Q Plot erstellen
# 'dist' gibt die Verteilung an (standardmäßig 'norm')
plt.figure(figsize=(8, 6))
stats.probplot(daten, dist="norm", plot=plt)

plt.title("Q-Q Plot gegen Normalverteilung")
plt.grid(True)
plt.show()

In [ ]:
import statsmodels.api as sm
import matplotlib.pyplot as plt

# 1. Q-Q Plot gegen die Normalverteilung
fig = sm.qqplot(daten, line='s')
plt.title("Statsmodels Q-Q Plot")
plt.show()

In [ ]:
from scipy.stats import expon

# 1. Daten generieren (Exponentialverteilt, z.B. scale=2.0)
np.random.seed(42)
daten_expon = np.random.exponential(scale=2.0, size=100)

# 2. Q-Q Plot gegen die theoretische Exponentialverteilung
# dist=stats.expon sagt der Funktion, womit sie vergleichen soll
fig = sm.qqplot(daten_expon, dist=expon, line="r")

plt.title("Q-Q Plot gegen Exponentialverteilung")
plt.xlabel("Theoretische Quantile (Expon)")
plt.ylabel("Stichproben Quantile")
plt.grid(True, alpha=0.3)
plt.show()

#### Shapiro-Wilk-Test (shapiro)
Dies ist der Spezialist für Normalverteilung. Er prüft, wie gut die Korrelation zwischen deinen Daten und den idealen Normalquantilen ist.
* **Stärke**: Er ist bei kleinen Stichproben (wie unseren $n=10$ Akkus) extrem leistungsstark und genau.
* **Hypothese**: $H_0$: Die Daten sind normalverteilt.

#### Kolmogorov-Smirnov-Test (kstest)
Dies ist der Generalist. Er vergleicht die kumulative Verteilungsfunktion (CDF) deiner Daten mit der einer theoretischen Verteilung.
* **Besonderheit**: Da er eigentlich für den Vergleich zweier völlig unabhängiger Verteilungen gedacht ist, müssen wir ihm Mittelwert und Standardabweichung unserer Daten als Referenz mitgeben (args=...).
* **Schwäche**: Bei sehr kleinen Stichproben ist er oft weniger empfindlich als der Shapiro-Wilk-Test.

In [ ]:
from scipy.stats import shapiro, kstest
import numpy as np

# Unsere Messdaten
data = [480, 490, 505, 485, 470, 495, 480, 510, 475, 480]

# 1. Shapiro-Wilk Test
# Liefert die Teststatistik und den p-Wert
p_sw = shapiro(data).pvalue

# 2. Kolmogorov-Smirnov Test (gegen Normalverteilung)
# Wir müssen den Mittelwert und die Std-Abw der Daten schätzen,
# damit der Test weiß, welche Normalverteilung er als Maßstab nehmen soll.
mu, std = np.mean(data), np.std(data, ddof=1)
p_ks = kstest(data, 'norm', args=(mu, std)).pvalue

print(f"Ergebnisse der Normalverteilungstests:")
print(f"--------------------------------------")
print(f"Shapiro-Wilk p-Wert: {p_sw:.4f}")
print(f"K-S Test p-Wert:     {p_ks:.4f}")

# Interpretation
alpha = 0.05
if p_sw > alpha:
    print("\nFazit (Shapiro): H0 kann nicht abgelehnt werden -> NV wird angenommen.")
else:
    print("\nFazit (Shapiro): H0 abgelehnt -> Daten weichen signifikant von NV ab.")



#### 🤼 Mann-Whitney-U-Test (U-Test)
Wenn wir zwei verschiedene Gruppen vergleichen wollen (z. B. Akkus von Hersteller A vs. Hersteller B), aber die Daten nicht normalverteilt sind oder wir Ausreißer haben, nutzen wir den Mann-Whitney-U-Test.

##### Das Prinzip: Rangsummen statt Mittelwerte
Ähnlich wie beim Vorzeichentest schauen wir nicht auf die nackten Zahlen, sondern wir werfen alle Werte in einen Topf und vergeben Ränge.
* Wenn die Akkus von Hersteller B systematisch höhere Ränge (längere Laufzeiten) besetzen als Hersteller A, dann ist das ein Zeichen für einen signifikanten Unterschied.
* Der Test ist das nicht-parametrische Gegenstück zum unabhängigen t-Test.

##### Hypothesen
* $H_0$: Die Verteilungen beider Gruppen sind identisch (keine Gruppe ist "besser").
* $H_1$: Eine Gruppe hat systematisch höhere Werte als die andere.

In [ ]:
from scipy.stats import mannwhitneyu

# Daten von zwei verschiedenen Herstellern
group_a = [480, 490, 505, 485, 470]
group_b = [510, 520, 530, 495, 515]

# Mann-Whitney U Test
# alternative='two-sided' ist Standard, 'less' oder 'greater' für gerichtete Hypothesen
stat, p_value = mannwhitneyu(group_a, group_b, alternative='two-sided')

print(f"Ergebnis des Mann-Whitney-U-Tests:")
print(f"----------------------------------")
print(f"U-Statistik: {stat}")
print(f"p-Wert:      {p_value:.4f}")

# Interpretation
alpha = 0.05
if p_value < alpha:
    print("\nFazit: p < 0,05 -> H0 abgelehnt.")
    print("Die Gruppen unterscheiden sich signifikant (eine Gruppe ist systematisch besser).")
else:
    print("\nFazit: p > 0,05 -> H0 beibehalten.")
    print("Keine signifikanten Unterschiede in der Rangfolge gefunden.")

#### Beispiel: Der Software-Update-Check (Chi-Quadrat-Unabhängigkeitstest)
Wir wollen wissen, ob ein neues Software-Update die Fehlerquote bei der Produktion beeinflusst oder ob die Schwankungen reiner Zufall sind.

| Software | OK | Ausschuss | **Summe** |
| :--- | :---: | :---: | :---: |
| **Alt** | 40 ($O_{11}$) | 10 ($O_{12}$) | **50** |
| **Neu** | 35 ($O_{21}$) | 15 ($O_{22}$) | **50** |
| **Summe** | **75** | **25** | **100** |

Die Idee: Was würden wir erwarten ($E$ = Expected)?Unter der Nullhypothese ($H_0$), dass das Update keinen Einfluss hat, müsste die Fehlerquote in beiden Gruppen identisch sein. Da insgesamt $25\ \%$ der Teile Ausschuss sind ($25/100$), würden wir bei 50 Teilen pro Gruppe jeweils 12,5 defekte Teile erwarten.

Die allgemeine Formel für jedes Feld lautet:

$$
E = \frac{\text{Zeilensumme} \cdot \text{Spaltensumme}}{N}
$$

Berechnung der Erwartungswerte ($E$) und Teil-Statistiken:

* Alt / OK: $\frac{50 \cdot 75}{100} = 37,5 \implies \frac{(40 - 37,5)^2}{37,5} = 0,1667$
* Alt / Ausschuss: $\frac{50 \cdot 25}{100} = 12,5 \implies \frac{(10 - 12,5)^2}{12,5} = 0,5000$
* Neu / OK: $\frac{50 \cdot 75}{100} = 37,5 \implies \frac{(35 - 37,5)^2}{37,5} = 0,1667$
* Neu / Ausschuss: $\frac{50 \cdot 25}{100} = 12,5 \implies \frac{(15 - 12,5)^2}{12,5} = 0,5000$

Teststatistik:
$$
\chi^2 = 0,1667 + 0,5000 + 0,1667 + 0,5000 = 1,3334
$$

Freiheitsgrade und Urteil
* Freiheitsgrade ($df$): $(\text{Zeilen} - 1) \cdot (\text{Spalten} - 1) = (2 - 1) \cdot (2 - 1) = 1$
* Kritischer Wert: Für $\alpha = 0,05$ und $df = 1$ liegt die Hürde bei 3,84.

Das Urteil: Da $1,33 < 3,84$ (bzw. $p > 0,05$), wird $H_0$ beibehalten. Das Update hat keinen statistisch nachweisbaren Einfluss auf die Fehlerquote.

⚠️ Wichtige Einschränkung: Wenn in der Erwartung ($E$) oben irgendwo eine Zahl unter 5 gestanden hätte, wäre die Chi-Quadrat-Annäherung zu ungenau. In diesem Fall müssten wir zwingend auf Fisher’s Exact Test ausweichen.

In [ ]:
import numpy as np
from scipy.stats import chi2_contingency

# 1. Die beobachteten Werte (O) in eine Matrix packen
# Zeile 1: Alt, Zeile 2: Neu
beobachtet = np.array([[40, 10],
                       [35, 15]])

# 2. Chi-Quadrat-Test durchführen
# Wir setzen correction=False, um die manuelle Formel 1:1 zu spiegeln
chi2_stat, p_val, df, erwartet = chi2_contingency(beobachtet, correction=False)

print(f"--- Software-Update Analyse ---")
print(f"Chi2-Statistik: {chi2_stat:.4f}")
print(f"p-Wert:         {p_val:.4f}")
print(f"Freiheitsgrade: {df}")
print(f"\nErwartete Werte (E):")
print(erwartet)
print(f"--------------------------------")

# 3. Entscheidungshilfe
alpha = 0.05
if p_val < alpha:
    print("Ergebnis: Signifikant! Das Update hat einen Effekt.")
else:
    print("Ergebnis: Nicht signifikant. Kein Effekt nachweisbar.")

#### 🎣 Fisher’s Exact Test: Die exakte Berechnung

Wenn wir sehr kleine Stichproben haben oder die Annahmen für den Chi-Quadrat-Test nicht erfüllt sind, berechnen wir die Wahrscheinlichkeit direkt über die hypergeometrische Verteilung.

##### Schritt 1: Die Wahrscheinlichkeit für GENAU diese Tabelle
Die Formel berechnet, wie wahrscheinlich es ist, exakt diese Verteilung bei den gegebenen Randsummen zu erhalten:

$$P = \frac{\binom{a+b}{a} \binom{c+d}{c}}{\binom{n}{a+c}} = \frac{(a+b)! (c+d)! (a+c)! (b+d)!}{n! a! b! c! d!}$$

Eingesetzt für unser Beispiel:

| Software | OK | Ausschuss | **Summe** |
| :--- | :---: | :---: | :---: |
| **Alt** | 40 ($O_{11}$) | 10 ($O_{12}$) | **50** |
| **Neu** | 35 ($O_{21}$) | 15 ($O_{22}$) | **50** |
| **Summe** | **75** | **25** | **100** |


$$P = \frac{50! \cdot 50! \cdot 75! \cdot 25!}{100! \cdot 40! \cdot 10! \cdot 35! \cdot 15!} \approx 0,0948$$

Die Wahrscheinlichkeit für exakt diese Konstellation liegt also bei ca. **9,48 %**.

---

##### Schritt 2: Das Protokoll der Entscheidung
Um den tatsächlichen p-Wert des Tests zu erhalten, addieren wir die Wahrscheinlichkeiten dieser Tabelle und aller noch „extremeren“ Tabellen auf.

* **Berechneter p-Wert:** $p = 0,3547$
* **Signifikanzniveau ($\alpha$):** $0,05$ (5 %)
* **Vergleich:** $0,3547 > 0,05$

**Das Urteil:** Wir können die Nullhypothese ($H_0$) **nicht ablehnen**. Es gibt keinen statistisch signifikanten Unterschied zwischen der alten und der neuen Software. Die beobachtete Abweichung (10 vs. 15 Fehler) ist mit einer Wahrscheinlichkeit von über 35 % reiner Zufall. Das Update hat die Fehlerquote also nicht nachweislich verändert.

In [ ]:
from scipy.stats import fisher_exact

# 1. Die Matrix erstellen (wie beim Chi-Quadrat)
# [Alt_OK, Alt_Ausschuss]
# [Neu_OK, Neu_Ausschuss]
tabelle = [[40, 10],
           [35, 15]]

# 2. Fisher's Exact Test durchführen
# oddsratio lassen wir links liegen, uns interessiert nur der p_value
_, p_value = fisher_exact(tabelle, alternative='two-sided')

print(f"--- Fisher's Exact Test Ergebnis ---")
print(f"p-Wert: {p_value:.4f}")
print(f"------------------------------------")

if p_value < 0.05:
    print("Ergebnis: Signifikant! (Unterschied vorhanden)")
else:
    print("Ergebnis: Nicht signifikant. (Zufall)")

# Einseitig bei Fisher heißt: Du prüfst nicht nur, ob es einen Zusammenhang gibt, sondern in welche Richtung. In SciPy geht es dabei um die Odds Ratio (Chancenverhältnis) der 2×2-Tabelle:
# 'two-sided': Gibt es irgendeinen Zusammenhang? (Odds Ratio ≠ 1)
# 'greater': Ist die Chance in der ersten Zeile größer? (Odds Ratio > 1)
# 'less': Ist die Chance in der ersten Zeile kleiner? (Odds Ratio < 1)

#### Beispiel: Die Präzisions-Fräse vs. der Prototyp
Stell dir vor, du arbeitest in der Qualitätssicherung. Du vergleichst zwei Maschinen, die Metallstifte mit einer Soll-Länge von 100 mm herstellen sollen.
* Maschine A (Der Profi): Eine seit Jahren kalibrierte Maschine. Sie arbeitet extrem präzise. Fast alle Stifte sind fast exakt gleich lang (winzige Varianz).
* Maschine B (Der Prototyp): Eine neue Maschine, die noch "zittert". Der Mittelwert scheint okay, aber die Werte streuen wie eine Schrotflinte (riesige Varianz). Zudem hast du vom Prototyp nur wenige Daten, vom Profi aber sehr viele.

#####  Warum der normale t-Test hier versagt:
Der Standard-t-Test geht davon aus, dass beide Maschinen „gleich stark zittern“ (Varianzgleichheit). Er wirft beide Varianzen in einen Topf (gepoolte Varianz).
* Wenn die Varianz von Maschine B riesig ist, "vergiftet" sie den Topf für Maschine A.
* Das Ergebnis: Der Standard-t-Test unterschätzt die Präzision von Gruppe A und liefert einen falschen p-Wert. Er erkennt Unterschiede nicht, die eigentlich da sind, oder erfindet welche.
* Die Lösung: Welch-t-Test. Der Welch-Test ist klüger: Er berechnet für jede Gruppe ihre eigene Varianz. Er passt die Freiheitsgrade ($df$) mathematisch an, um das Ungleichgewicht auszugleichen.

In [ ]:
import numpy as np
from scipy import stats

# 1. Daten generieren
np.random.seed(42)

# Gruppe A: n=100, Mittelwert=100, Standardabweichung=1 (Präzise & Viele Daten)
maschine_a = np.random.normal(loc=100, scale=1, size=100)

# Gruppe B: n=20, Mittelwert=102, Standardabweichung=10 (Ungenau & Wenige Daten)
# Der wahre Unterschied ist 2mm.
maschine_b = np.random.normal(loc=102, scale=10, size=20)

# 2. t-Tests durchführen
# Standard Student's t-Test (equal_var=True)
t_stat_std, p_val_std = stats.ttest_ind(maschine_a, maschine_b, equal_var=True)

# Welch-t-Test (equal_var=False)
t_stat_welch, p_val_welch = stats.ttest_ind(maschine_a, maschine_b, equal_var=False)

# 3. Ergebnisse ausgeben
print(f"--- Mittelwerte ---")
print(f"Maschine A: {np.mean(maschine_a):.2f} | Maschine B: {np.mean(maschine_b):.2f}")
print(f"-------------------\n")

print(f"STUDENT'S T-TEST (Falsche Annahme gleicher Varianz):")
print(f"p-Wert: {p_val_std:.4f}")

print(f"\nWELCH-T-TEST (Korrekte Behandlung ungleicher Varianz):")
print(f"p-Wert: {p_val_welch:.4f}")

print(f"\n--- URTEIL ---")
if p_val_std > 0.05 and p_val_welch <= 0.05:
    print("Oha! Der Standard-t-Test hätte den Unterschied übersehen (Nicht signifikant),")
    print("aber der Welch-t-Test erkennt ihn korrekt (Signifikant)!")
elif p_val_std <= 0.05 and p_val_welch > 0.05:
    print("Der Standard-t-Test schreit 'Signifikant!', aber der Welch-Test")
    print("bleibt vorsichtig. Der Student-Test war hier zu optimistisch.")
else:
    print("Beide sind sich einig, aber die p-Werte unterscheiden sich trotzdem.")

| Test-Funktion | Name | Datenquelle | Vergleich | Typisches Szenario |
| :--- | :--- | :--- | :--- | :--- |
| **`ttest_1samp`** | Ein-Stichproben-Test | **1 Gruppe** | Stichprobe vs. **Fixwert** ($\mu$) | "Entspricht der Inhalt (Soll: 500g) meinen Messungen?" |
| **`ttest_ind`** | Unabhängiger Test | **2 fremde Gruppen** | Gruppe A vs. **Gruppe B** | "Ist Maschine A produktiver als Maschine B?" (Unterschiedliche Geräte) |
| **`ttest_rel`** | Gepaarter Test | **1 Gruppe (2 Messungen)** | Messung 1 vs. **Messung 2** | "Wie ist die Leistung desselben Rechners **vor** und **nach** einem Update?" |

In [ ]:
from scipy.stats import ttest_1samp, ttest_ind, ttest_rel

# 1. Ein-Stichproben-Test (Gegen einen Standard)
# Hat der Akku wirklich 5000 mAh?
messungen = [4950, 5020, 4890, 4910]
t1, p1 = ttest_1samp(messungen, popmean=5000)

# 2. Unabhängiger Test (Zwei fremde Gruppen)
# Welch-Test nutzen (equal_var=False) ist der sicherste Standard!
gruppe_a = [10.2, 11.1, 10.5]
gruppe_b = [12.4, 13.0, 12.1]
t2, p2 = ttest_ind(gruppe_a, gruppe_b, equal_var=False)

# 3. Gepaarter Test (Dasselbe Objekt zweimal gemessen)
# Leistung derselben 3 Rechner vor und nach Optimierung
vorher = [15.0, 14.5, 16.2]
nachher = [12.1, 11.8, 12.5]
t3, p3 = ttest_rel(vorher, nachher)

print(f"p-Werte: 1-Samp: {p1:.4f} | Unabh: {p2:.4f} | Gepaart: {p3:.4f}")

```
Was vergleichst du?
│
├── Gruppe vs. Fixwert
│   └── normalverteilt ODER n ≥ 30?
│       ├── JA   → ttest_1samp(x, popmean)
│       └── NEIN → wilcoxon(x - popmean)
│
├── Gruppe vs. Gruppe
│   ├── unabhängig
│   │   ├── 2 Gruppen
│   │   │   └── normalverteilt ODER n ≥ 30 je Gruppe?
│   │   │       ├── JA   → ttest_ind(g1, g2, equal_var=False)   (Welch)
│   │   │       └── NEIN → mannwhitneyu(g1, g2)
│   │   └── >2 Gruppen
│   │       └── normalverteilt ODER n ≥ 30 je Gruppe?
│   │           ├── JA   → f_oneway(g1, g2, g3)  → Post-hoc: tukey_hsd
│   │           └── NEIN → kruskal(g1, g2, g3)   → Post-hoc: Dunn-Test
│   │
│   └── abhängig (gleiche Personen)
│       ├── 2 Messungen
│       │   └── Differenzen normalverteilt ODER n ≥ 30?
│       │       ├── JA   → ttest_rel(vorher, nachher)
│       │       └── NEIN → wilcoxon(vorher, nachher)
│       └── >2 Messungen
│           ├── JA   → AnovaRM (statsmodels)
│           └── NEIN → friedmanchisquare(t1, t2, t3)
│
└── Häufigkeiten / Kategorien
    ├── 1 Variable vs. erwartete Verteilung → chisquare(beob, erw)
    └── 2 Variablen (Kreuztabelle)          → chi2_contingency(tabelle)
                                             (2×2 & erwartet < 5 → fisher_exact)
```


| Ziel des Tests                               | Datenbasis | Parametrischer Test <br>*(Normalverteilung vorausgesetzt)* | Nicht-parametrischer Test <br>*(Robust gegen Ausreißer/schiefe Daten)* | Typisches Praxis-Szenario |
|:---------------------------------------------| :--- | :--- | :--- | :--- |
| **1 Gruppe gegen Fixwert**                   | 1 Stichprobe vs. theoretischer Sollwert | **`ttest_1samp(x, popmean)`** <br>*(Ein-Stichproben t-Test)* | **`wilcoxon(x - popmean)`** <br>*(Wilcoxon-Vorzeichen-Rangtest)* | "Haben die Säcke im Schnitt wirklich die aufgedruckten 50 kg?" |
| **2 unabhängige Gruppen**                    | 2 komplett getrennte Gruppen | **`ttest_ind(g1, g2)`** <br>*(Unabhängiger t-Test)* | **`mannwhitneyu(g1, g2)`** <br>*(Mann-Whitney-U-Test)* | "Klicken Männer (`g1`) häufiger auf die Werbung als Frauen (`g2`)?" |
| **2 abhängige Messungen**                    | 1 Gruppe, aber 2x gemessen (paired) | **`ttest_rel(vorher, nachher)`** <br>*(Gepaarter t-Test)* | **`wilcoxon(vorher, nachher)`** <br>*(Gepaarter Wilcoxon-Test)* | "Ist der Blutdruck derselben Patienten **nach** der Tabletteneinnahme gesunken?" |
| **Mehr als 2 unabhängige Gruppen (spoiler)** | 3 oder mehr getrennte Gruppen | **`f_oneway(g1, g2, g3)`** <br>*(Einfaktorielle ANOVA)* | **`kruskal(g1, g2, g3)`** <br>*(Kruskal-Wallis-Test)* | "Unterscheidet sich die Ladezeit der Website bei Browser A, B und C?" |
| **Häufigkeiten / Kategorien**                | Gezählte Daten (z.B. Ja/Nein, Farben) | *Nicht anwendbar (da keine stetigen Mittelwerte)* | **`chi2_contingency(tabelle)`** <br>*(Chi-Quadrat-Unabhängigkeitstest)* | "Gibt es einen Zusammenhang zwischen der Automarke und der gewählten Farbe?" |

### Entscheidungsbaum: Vergleich von 2 Stichproben (A vs. B)

Wenn du testen willst, ob sich zwei Gruppen statistisch unterscheiden, hängt die Wahl des Tests von der **Normalverteilung** und der **Stichprobengröße (n)** ab.

---

#### 1. Der Fahrplan (Schritt für Schritt)

1. **Normalverteilung prüfen:** Du jagst **beide** Gruppen separat durch den Shapiro-Wilk-Test (`scipy.stats.shapiro`).
2. **Die "ZGS-Ausnahme" prüfen:** Ist deine Stichprobe pro Gruppe groß genug ($n > 30$)?
    * Wenn ja, rettet dich der Zentrale Grenzwertsatz. Du darfst meistens trotzdem den t-Test nutzen.
        * Ausreißer prüfen (z. B. mit Boxplots, Histogrammen oder IQR-Regel): ZGS hilft bei großen Stichproben gegen leichte bis mäßige Verletzungen der Normalverteilungsannahme. Er schützt aber nicht vor extremen Ausreißern, da diese den Mittelwert und die Varianz stark beeinflussen können.
        * IQR-Regel: $x<Q_1−3 \cdot IQR$ oder $x>Q_3+3 \cdot IQR$
        * Es gibt keine feste Grenzanzahl. Schon einzelne extreme Werte können den Mittelwert stark verschieben. Entscheidend ist ihr Einfluss: Test mit und ohne Ausreißer rechnen. Ändert sich das Ergebnis deutlich, besser einen robusten Test nutzen (Mann-Whitney-U oder getrimmter t-Test).
    * Wenn nein ($n < 30$), musst du streng nach der Normalverteilung gehen.

---

#### 2. Die beiden Pfade

##### Pfad A: Beide Gruppen sind normalverteilt (oder $n > 30$)
-> Du nutzt den **unabhängigen t-Test**.
* **Empfehlung:** Nutze immer den **Welch-t-Test** (`equal_var=False`), da er nicht verlangt, dass die Varianzen der beiden Gruppen gleich sind.
* **Python:** `scipy.stats.ttest_ind(gruppe_a, gruppe_b, equal_var=False)`

##### Pfad B: Mindestens eine Gruppe ist NICHT normalverteilt (und $n < 30$)
-> Du nutzt den **Mann-Whitney-U-Test** (auch Wilcoxon-Rangsummentest genannt).
* **Wie er funktioniert:** Er vergleicht nicht die echten Mittelwerte, sondern wandelt die Daten in Ränge (Platzierungen) um. Er prüft, ob die Werte einer Gruppe systematisch höher liegen als die der anderen.
* **Python:** `scipy.stats.mannwhitneyu(gruppe_a, gruppe_b, alternative='two-sided')`

---

#### 🆚 Direktvergleich im Spickzettel

| Kriterium | Welch-t-Test (`ttest_ind`) | Mann-Whitney-U-Test (`mannwhitneyu`) |
| :--- | :--- | :--- |
| **Typ** | Parametrisch (strenger) | Nicht-parametrisch (robuster) |
| **Voraussetzung** | Normalverteilung (bei kleinem $n$) | Keine (Verteilung ist egal) |
| **Verglichen wird...** | Der Unterschied der **Mittelwerte** | Der Unterschied der **Ränge / Mediane** |
| **Anfälligkeit** | Reagiert empfindlich auf Ausreißer | Extrem robust gegen Ausreißer |

---


In [ ]:
#### Minimal-Code für dein Notebook: A vs B

import numpy as np
from scipy.stats import shapiro, ttest_ind, mannwhitneyu

ALPHA = 0.05   # Signifikanzniveau
N_MIN = 30     # ab hier greift der Zentrale Grenzwertsatz (Faustregel)


def extreme_ausreisser(x, k=3):
    """Werte außerhalb von Q1 - k*IQR bzw. Q3 + k*IQR."""
    x = np.asarray(x)
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    maske = (x < q1 - k * iqr) | (x > q3 + k * iqr)
    return x[~maske], x[maske]          # (ohne Ausreißer, Ausreißer)


def verteilung_ok(x, name):
    """Mittelwert-Test zulässig, wenn n >= 30 ODER Shapiro-Wilk nicht signifikant."""
    n = len(x)
    if n >= N_MIN:
        print(f" {name}: n = {n} >= {N_MIN} -> ZGS greift")
        return True
    _, p_norm = shapiro(x)
    ok = p_norm > 0.05
    print(f" {name}: n = {n} < {N_MIN}, Shapiro p = {p_norm:.4f} -> "
          f"{'normalverteilt' if ok else 'NICHT normalverteilt'}")
    return ok


def vergleiche_gruppen(gruppe_a, gruppe_b):
    a, b = np.asarray(gruppe_a), np.asarray(gruppe_b)

    # 1. Vorab-Checks
    print("=" * 60)
    print(" VORAB-CHECKS")
    print("-" * 60)
    vert_ok = verteilung_ok(a, "Gruppe A") & verteilung_ok(b, "Gruppe B")

    a_ohne, a_aus = extreme_ausreisser(a)
    b_ohne, b_aus = extreme_ausreisser(b)
    n_aus = len(a_aus) + len(b_aus)
    print(f" Extreme Ausreißer (3·IQR): A = {len(a_aus)}, B = {len(b_aus)}")

    # 2. Test-Auswahl
    if not vert_ok:
        test_name = "Mann-Whitney-U-Test (kleine Stichprobe, nicht normalverteilt)"
        _, p_value = mannwhitneyu(a, b, alternative="two-sided")

    elif n_aus == 0:
        test_name = "Welch-t-Test"
        _, p_value = ttest_ind(a, b, equal_var=False)

    else:
        # Sensitivitätsanalyse: Ändert sich die Entscheidung ohne Ausreißer?
        _, p_mit = ttest_ind(a, b, equal_var=False)
        _, p_ohne = ttest_ind(a_ohne, b_ohne, equal_var=False)
        print(f" Sensitivität: p mit Ausreißern = {p_mit:.4f}, "
              f"p ohne = {p_ohne:.4f}")

        if (p_mit < ALPHA) == (p_ohne < ALPHA):
            test_name = "Welch-t-Test (Ausreißer ändern die Entscheidung nicht)"
            p_value = p_mit                       # Ergebnis auf ALLEN Daten
        else:
            test_name = "Mann-Whitney-U-Test (Ausreißer kippen das t-Test-Ergebnis)"
            _, p_value = mannwhitneyu(a, b, alternative="two-sided")

    # 3. Auswertung und Interpretation
    print("=" * 60)
    print(f" Angewendeter Test:  {test_name}")
    print(f" Mittelwert A / B:   {a.mean():.2f} / {b.mean():.2f}")
    print(f" Median     A / B:   {np.median(a):.2f} / {np.median(b):.2f}")
    print(f" p-Wert:             {p_value:.4f}")
    print("-" * 60)

    if p_value < ALPHA:
        print(" ERGEBNIS: STATISTISCH SIGNIFIKANT")
        print(f" p ({p_value:.4f}) < alpha ({ALPHA}) -> H0 wird abgelehnt.")
        print(" -> Die Gruppen unterscheiden sich statistisch signifikant.")
        print("    (Ob der Unterschied praktisch relevant ist, zeigt die Effektstärke.)")
    else:
        print(" ERGEBNIS: NICHT SIGNIFIKANT")
        print(f" p ({p_value:.4f}) >= alpha ({ALPHA}) -> H0 wird beibehalten.")
        print(" -> Kein Unterschied nachweisbar. Das beweist NICHT, dass es")
        print("    keinen gibt (evtl. zu kleine Stichprobe / zu geringe Teststärke).")
    print("=" * 60)

    return test_name, p_value



rng = np.random.default_rng(42)

print("\n### Beispiel 1: große Gruppen, ein extremer Ausreißer")
gruppe_a = np.append(rng.normal(50, 10, 40), 250)
gruppe_b = rng.normal(55, 10, 40)
vergleiche_gruppen(gruppe_a, gruppe_b)

print("\n### Beispiel 2: kleine, schiefe Gruppen")
gruppe_a = rng.exponential(5, 12)
gruppe_b = rng.exponential(10, 12)
vergleiche_gruppen(gruppe_a, gruppe_b)